In [1]:
%%capture
!pip install yadisk

In [2]:
import cv2
import torch
import numpy as np
import json
from pathlib import Path
from tqdm import tqdm
import yadisk
import os

from transformers import AutoProcessor, AutoModel

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
client = yadisk.Client(
    token=Path('/content/drive/MyDrive/Colab Notebooks/ai msc/token').read_text(encoding='utf-8')
  )

In [ ]:
%%capture
processor = AutoProcessor.from_pretrained("microsoft/xclip-base-patch32", device='cuda')
model = AutoModel.from_pretrained("microsoft/xclip-base-patch32").to('cuda')

In [ ]:
test = json.load(
    Path('/content/drive/MyDrive/Colab Notebooks/ai msc/test.json').open('r', encoding='utf-8')
)

In [ ]:
translations = json.load(
    Path('/content/drive/MyDrive/Colab Notebooks/ai msc/translations.json').open('r', encoding='utf-8')
)

In [ ]:
def read_vid(vid, start, end):
    out = []
    cur = start
    frame_rate = (end-start) / 8
    while cur < end:
        vid.set(cv2.CAP_PROP_POS_MSEC, cur*1000)
        out.append(vid.read()[1][:, :, ::-1])
        cur += frame_rate
    return out

In [4]:
res = json.load(
    Path('/content/drive/MyDrive/Colab Notebooks/ai msc/eval_res.json').open('r', encoding='utf-8')
)

In [ ]:
for i, cut in enumerate(test):

  if i < len(res):
    continue

  client.download("disk:/SLR Project/" + cut['video'], cut['video'])

  vids = read_vid(
          cv2.VideoCapture(cut['video']),
          cut['start'],
          cut['end']
      )[:8]

  os.remove(cut['video'])

  inputs = processor(
    text=translations,
    videos=vids,
    return_tensors="pt",
    padding=True
  ).to('cuda')

  # forward pass
  with torch.no_grad():
      outputs = model(**inputs)

  logits_per_video = outputs.logits_per_video  # this is the video-text similarity score
  probs = logits_per_video.softmax(dim=1)  # we can take the softmax to get the label probabilities

  res.append(
      ((-probs).argsort() + 1)[0, i].item()
  )

  json.dump(res, Path('/content/drive/MyDrive/Colab Notebooks/ai msc/eval_res.json').open('w', encoding='utf-8'))

/usr/local/lib/python3.10/dist-packages/transformers/image_processing_utils.py:41: UserWarning: The following named arguments are not valid for `VideoMAEImageProcessor.preprocess` and were ignored: 'padding'
  return self.preprocess(images, **kwargs)


In [6]:
res = np.array(res)

In [10]:
print(f'''
MRR: {round((1/res).mean(), 5)}
Hit@1: {round((res==1).sum() / len(res), 5)}
Hit@100: {round((res<=100).sum() / len(res), 5)}
''')


MRR: 0.00321
Hit@1: 0.00036
Hit@100: 0.03801

